# Kriging workflow

Kriging turns **scattered point observations** into a **continuous surface plus an uncertainty estimate**. The
pipeline is always the same:

**explore spatial structure → fit a variogram → krige onto a grid → validate honestly.**

Everything hangs off `Samples`, a `FeatureCollection` subclass, so the column to interpolate is always a method
argument. This notebook walks the full pipeline on a synthetic rain-gauge field.

In [ ]:
%matplotlib inline
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from geostatista import Samples

# A smooth spatial field (two sinusoids) sampled at 80 random gauge locations, in metres (UTM 33N).
rng = np.random.default_rng(0)
xy = rng.uniform(0.0, 100.0, (80, 2))
rain = np.sin(xy[:, 0] / 25.0) * np.cos(xy[:, 1] / 25.0) * 10.0 + 20.0
gdf = gpd.GeoDataFrame({"rain": rain}, geometry=[Point(x, y) for x, y in xy], crs="EPSG:32633")

samples = Samples(gdf)          # a Samples *is-a* FeatureCollection
samples.head()

`Samples` exposes two convenience accessors that validate the geometry, drop `NaN` rows, and hand back plain
NumPy — the exact `(coords, values)` every geostatistics routine consumes.

In [ ]:
print("values[:5] :", np.round(samples.values("rain")[:5], 2))
print("coords[:3] :\n", np.round(samples.coords("rain")[:3], 2))

## 1. Empirical variogram — look before you krige

The empirical variogram bins pairwise squared differences by lag distance: it shows how quickly the field
de-correlates with separation. `semivariance` rises from near zero and flattens out around the *sill*.

In [ ]:
vg = samples.variogram("rain", n_lags=12)     # estimator="matheron" by default
vg.to_dataframe()                              # lag / semivariance / count

Plotting needs the `viz` extra (cleopatra). Before fitting, `Variogram.plot` shows just the empirical cloud.

In [ ]:
fig, ax = vg.plot(title="Empirical variogram — rainfall")

## 2. Fit a model — the (nugget, sill, range) triple kriging needs

Kriging needs a valid, positive-definite model, not the raw cloud. Choose `spherical`, `exponential`, `gaussian`,
or `matern`. Fitting sets `.nugget`, `.sill`, and `.range_`, and returns the variogram for chaining.

In [ ]:
vg.fit(model="spherical")
print(vg)
print("gamma(0)  =", round(float(vg.predict(0.0)), 3))
print("gamma(30) =", round(float(vg.predict(30.0)), 3), " (near the sill)")

Now `Variogram.plot` overlays the fitted model curve on the empirical cloud.

In [ ]:
fig, ax = vg.plot(title="Spherical fit")

## 3. Krige onto a grid — a 2-band surface (estimate + variance)

`krige` (an alias for `interpolate_to_raster(method="kriging")`) returns a `KrigedSurface`: a 2-band `Dataset`
where **band 0 is the estimate** and **band 1 is the kriging variance**. `n_neighbors` bounds the moving
neighborhood around each target cell — the key to scaling past a few thousand samples.

In [ ]:
surface = samples.krige("rain", vg, cell_size=5.0, n_neighbors=24)
arr = np.asarray(surface.read_array())
print("bands :", surface.band_count, " epsg :", surface.epsg, " shape :", arr.shape)
print("estimate range :", round(arr[0].min(), 2), "->", round(arr[0].max(), 2))
print("variance >= 0 everywhere :", bool(np.all(arr[1] >= -1e-9)))

Map the estimate and — the reason to prefer kriging over IDW — the uncertainty band. The variance is lowest near
the gauges and grows where the surface is extrapolated.

In [ ]:
est_glyph = surface.estimate.plot()      # band 0 — the interpolated surface

In [ ]:
var_glyph = surface.variance.plot()      # band 1 — where the estimate is least certain

The surface is **self-describing**: the producing variogram is stored as typed attributes and written to the
raster's metadata tags (`GS_*`), so a saved GeoTIFF round-trips its own provenance.

In [ ]:
# The producing variogram is carried as typed attributes and mirrored into the raster metadata tags, so
# `surface.to_file("rain.tif")` yields a 2-band GeoTIFF that describes its own provenance.
print("provenance:", surface.model, "nugget=%.3g" % surface.nugget,
      "sill=%.3g" % surface.sill, "range=%.3g" % surface.range_)
{k: v for k, v in surface.meta_data.items() if k.startswith("GS_")}

## 4. The convenient variants — auto-fit, a template grid, and IDW

You can skip the explicit `Variogram` and pass a **model name** to auto-fit it internally; align the output grid
to an existing `Dataset` with `template=`; or fall back to inverse-distance weighting with `method="idw"` (which
delegates to pyramids — see [ADR 0001](../adr/0001-pyramids-geostatista-boundary.md)).

In [ ]:
# (a) auto-fit an exponential model by name — no explicit Variogram object
auto = samples.krige("rain", "exponential", cell_size=5.0)
print("auto-fit surface :", auto.model, " bands :", auto.band_count)

# (b) align a new kriging to the grid of an existing Dataset
aligned = samples.krige("rain", vg, template=surface.estimate)
print("template-aligned grid :", np.asarray(aligned.read_array()).shape[1:], "==", arr.shape[1:])

# (c) inverse-distance weighting instead of kriging (single band, no variance)
idw = samples.interpolate_to_raster("rain", method="idw", cell_size=5.0)
print("idw surface bands :", idw.band_count)

### Under the hood — the `OrdinaryKriging` engine

`Samples.krige` drives an `OrdinaryKriging` engine you can also use directly for single-point predictions.

In [ ]:
from geostatista import OrdinaryKriging

engine = OrdinaryKriging(samples.coords("rain"), samples.values("rain"), vg, n_neighbors=16)
estimate, variance = engine.predict_point(np.array([50.0, 50.0]))
print("estimate at (50, 50) :", round(estimate, 3), " kriging variance :", round(variance, 3))

## 5. Validate honestly — leave-one-out cross-validation

Cross-validation drops each point in turn, krige-predicts it from the rest, and reports the errors. For a
well-specified variogram the **mean standardized error ≈ 0** and the **standardized RMSE ≈ 1**.

In [ ]:
cv = samples.cross_validate("rain", vg, n_neighbors=None)   # None -> exact global solve
cv.head()

In [ ]:
{k: round(v, 4) for k, v in cv.attrs["summary"].items()}

That is the whole pipeline: **variogram → fit → krige (estimate + variance) → cross-validate**, all as methods on
`Samples`. See the [variogram-models notebook](03_variogram_models.ipynb) for choosing between models, and the
[spatial-autocorrelation notebook](02_spatial_autocorrelation.ipynb) for the other half of the package.